# 06 — Reprodukowalność danych i ścieżka rekonstrukcji

**Cel.** Notebook dokumentuje zawartość pozostałych podkatalogów
`appendix/` (D — schema bazy, F — środowisko, G — surowe pliki per-run,
H — manifest 240 runów) oraz `CITATION.md`. Razem z notebookiem 05
stanowi *kompletny* przegląd całego materiału załącznika cyfrowego.

**Kontekst.** Reprodukowalność wyników metaheurystyk wymaga (López-Ibáñez
et al. 2021):
1. Identycznego stanu repozytorium (commit hash — `CITATION.md`).
2. Identycznych wersji pakietów (`F_environment/conda_env_export.yaml`).
3. Surowych danych per-run (`G_per_run_seeds/`) lub procedury ich
   wygenerowania (`E_configs/` + `H_run_manifest.csv`).
4. Schematu agregatu (`D_database_schema/schema.sql`).

Notebook iteruje po każdym z tych elementów i pokazuje, jak je
wykorzystać.

In [ ]:
import prepare_notebook  # noqa: F401

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

PROJECT_ROOT = Path(prepare_notebook.project_root)
APPENDIX = PROJECT_ROOT / "appendix"
pd.set_option("display.max_colwidth", 80)

## CITATION — commit referencyjny

Zawartość `appendix/CITATION.md` — pełen identyfikator wersji kodu, którą
wykorzystano do wygenerowania wszystkich artefaktów w załączniku.

In [ ]:
display(Markdown((APPENDIX / "CITATION.md").read_text(encoding="utf-8")))

## H — manifest 240 runów i sanity-check kompletności

`H_run_manifest.csv` zawiera po jednym wierszu na każdy zrealizowany run
eksperymentu wraz ze statusem agregacji. Notebook weryfikuje, czy
uzyskano pełny szablon **4 algorytmów × 2 środowiska × 30 ziaren = 240**.

In [ ]:
manifest = pd.read_csv(APPENDIX / "H_run_manifest.csv")
print(f"Liczba wierszy: {len(manifest)}")
print(f"Status agregacji: {dict(manifest['aggregation_status'].value_counts())}")

# Pełny szablon eksperymentu — 4 × 2 × 30 = 240.
expected = pd.MultiIndex.from_product(
    [sorted(manifest["optimizer_algo"].unique()),
     sorted(manifest["environment"].unique()),
     sorted(manifest["seed"].unique())],
    names=["optimizer_algo", "environment", "seed"],
)
actual = manifest.set_index(["optimizer_algo", "environment", "seed"]).index
missing = expected.difference(actual)
print(f"Oczekiwane kombinacje: {len(expected)}, brakujące: {len(missing)}")
if len(missing):
    display(pd.DataFrame(list(missing), columns=expected.names))

# Heatmapa kompletności runów (zliczanie ziaren per (alg, env)).
pivot = manifest.pivot_table(index="optimizer_algo", columns="environment",
                              values="seed", aggfunc="count", fill_value=0)
display(pivot)

## D — schema bazy `analysis.db` i diagram ERD

Pełna baza analityczna (`analysis.db`, ≈ 861 MB) **nie** jest dołączona —
może zostać zrekonstruowana z surowych plików w `G_per_run_seeds/` przy
pomocy `src.analysis.ExperimentAggregator`. Schemat (21 tabel, z czego 11
cytowanych w pracy, oraz 5 widoków) jest udokumentowany w trzech plikach.

* `schema.sql` — pełna definicja SQL DDL.
* `ERD.md` — uproszczony diagram Mermaid (relacje między 11 tabelami
  cytowanymi w pracy).
* `views.md` — opis 5 widoków (`vw_run_summary`, `vw_seed_summary`,
  `vw_global_summary`, `vw_run_online_summary`, `vw_algo_cross_sim_comparison`).

Poniżej renderujemy te trzy dokumenty (Mermaid jest natywnie obsługiwany
w JupyterLab ≥ 3.5).

In [ ]:
display(Markdown((APPENDIX / "D_database_schema" / "ERD.md").read_text(encoding="utf-8")))

In [ ]:
# Schema SQL — pokazujemy nagłówek i listę zdefiniowanych tabel/widoków.
schema_path = APPENDIX / "D_database_schema" / "schema.sql"
schema_text = schema_path.read_text(encoding="utf-8")
tables = [ln for ln in schema_text.splitlines() if ln.strip().startswith("CREATE TABLE")]
views = [ln for ln in schema_text.splitlines() if ln.strip().startswith("CREATE VIEW")]
indexes = [ln for ln in schema_text.splitlines() if ln.strip().startswith("CREATE INDEX")]
print(f"Tabele ({len(tables)}):")
for t in tables: print("  ", t.strip())
print(f"\nWidoki ({len(views)}):")
for v in views: print("  ", v.strip())
print(f"\nIndeksy: {len(indexes)} szt. — pełna lista w schema.sql")
print(f"Plik: {schema_path.relative_to(PROJECT_ROOT)}  ({schema_path.stat().st_size / 1024:.1f} KB)")

In [ ]:
display(Markdown((APPENDIX / "D_database_schema" / "views.md").read_text(encoding="utf-8")))

## F — środowisko wykonawcze (snapshot + reprodukcja)

Pełna specyfikacja Python+conda+pip. `environment.yaml` wystarczy do
stworzenia środowiska na czystej maszynie (`conda env create -f …`);
`conda_env_export.yaml` zawiera wszystkie wersje i hash'e — dokładny
snapshot do bit-exact porównywania.

Notebook wykrywa, czy aktywne środowisko wykonawcze ma zbieżne wersje
krytycznych pakietów (pymoo, mealpy, pybullet, numba, numpy, scipy).

In [ ]:
import yaml
from importlib.metadata import PackageNotFoundError, version

# Wyciągnięcie wersji z `environment.yaml` (mieszany conda + pip).
env_yaml = yaml.safe_load((APPENDIX / "F_environment" / "environment.yaml").read_text())

declared = {}
for entry in env_yaml.get("dependencies", []):
    if isinstance(entry, str) and "=" in entry and not entry.startswith("_"):
        # `numpy=1.26` lub `python=3.10`
        name, _, ver = entry.partition("=")
        declared[name.strip().split("::")[-1]] = ver.split("=")[0]
    elif isinstance(entry, dict) and "pip" in entry:
        for pip_dep in entry["pip"]:
            if isinstance(pip_dep, str) and "==" in pip_dep:
                n, v = pip_dep.split("==")
                declared[n.strip()] = v.strip()

CRITICAL = ["numpy", "scipy", "pandas", "matplotlib", "pymoo", "mealpy",
             "pybullet", "numba", "hydra-core", "h5py"]

rows = []
for pkg in CRITICAL:
    try:
        installed = version(pkg)
    except PackageNotFoundError:
        installed = "n/a"
    rows.append({"package": pkg,
                  "declared_in_environment.yaml": declared.get(pkg, "(brak)"),
                  "locally_installed": installed})
display(pd.DataFrame(rows))

## G — surowe pliki per-run i przykładowa rekonstrukcja konwergencji

240 katalogów (`<optimizer>_<env>_<avoidance>_seed<N>/`) zawiera surowe
wyjścia per-run, z których można zrekonstruować pełną analizę:

| Plik | Format | Cytowane metryki |
|---|---|---|
| `optimization_history/optimization_history.h5` | HDF5 | M1, M2, M3, M9, M10 |
| `online_optimization.csv` | CSV | M6, M7, M11, M13 |
| `convergence_traces.csv` | CSV | M12 |
| `evasion_events.csv` | CSV | M5, M8 (faza klasyfikacja) |

Poniżej rekonstruujemy krzywą konwergencji offline (M10) dla wybranego
runa bezpośrednio z `optimization_history.h5` i porównujemy z
zagregowanym subsetem z `A_metrics/iteration_metrics_subset.csv`.

In [ ]:
# Edytuj `EXAMPLE_RUN`, by spojrzeć na inny run (np. `ssa_urban_ssa_seed12`).
EXAMPLE_RUN = "msffoa_forest_msffoa_seed1"

run_dir = APPENDIX / "G_per_run_seeds" / EXAMPLE_RUN
assert run_dir.is_dir(), f"Brak runa: {run_dir}"
print(f"Katalog: {run_dir.relative_to(PROJECT_ROOT)}")
print("Pliki:", sorted(p.name for p in run_dir.rglob("*") if p.is_file()))

In [ ]:
import h5py

h5_path = run_dir / "optimization_history" / "optimization_history.h5"
with h5py.File(h5_path, "r") as f:
    print(f"Klucze w h5: {list(f.keys())}")
    objectives = f["objectives_matrix"][...]          # (n_gen, pop, n_obj)
    feasible = f["feasible_mask"][...]                # (n_gen, pop)
print(f"Kształt objectives: {objectives.shape}")

# Best-so-far feasible (per iteration_metrics.populate_iteration_metrics).
summed = objectives.sum(axis=-1)
summed = np.where(feasible, summed, np.inf)
best_per_gen = summed.min(axis=1)
raw_best_so_far = np.minimum.accumulate(best_per_gen)

# Porównanie z subsetem A_metrics.
iters_subset = pd.read_csv(APPENDIX / "A_metrics" / "iteration_metrics_subset.csv")
subset_for_run = iters_subset[iters_subset["run_id"] == EXAMPLE_RUN]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(raw_best_so_far, label="raw h5 (z G_per_run_seeds)", linewidth=1.5)
if not subset_for_run.empty:
    ax.plot(subset_for_run["iteration"], subset_for_run["best_so_far"],
            label="agregat z A_metrics/iteration_metrics_subset.csv",
            linestyle="--", linewidth=1.2)
ax.set_xlabel("Iteracja"); ax.set_ylabel("$\\sum F_i$ (feasible best-so-far)")
ax.set_title(f"M10 — rekonstrukcja konwergencji offline\n{EXAMPLE_RUN}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Przykład wyciągnięcia detali per trygera uniku — `online_optimization.csv`.
online_path = run_dir / "online_optimization.csv"
if online_path.exists() and online_path.stat().st_size > 0:
    online = pd.read_csv(online_path)
    cols = [c for c in ["drone_id", "trigger_time", "algorithm", "status",
                          "best_fitness", "plan_arc_length_m", "chosen_axis",
                          "outcome", "time_to_rejoin_s"] if c in online.columns]
    print(f"Liczba trygerów uniku w {EXAMPLE_RUN}: {len(online)}")
    display(online[cols].head(10))
else:
    print(f"[INFO] Brak `online_optimization.csv` w {run_dir.name} — run bez uników.")

## Procedura pełnej rekonstrukcji eksperymentu

Aby zrekonstruować pełną bazę `analysis.db` z surowych plików w
`G_per_run_seeds/`:

```python
from src.analysis.ExperimentAggregator import ExperimentAggregator
ExperimentAggregator().aggregate('appendix/G_per_run_seeds/')
# Wynik: results/<reconstructed_exp>/analysis.db
```

Aby ponownie wygenerować raz wszystkie 240 runów:

```bash
# 1. Sprawdź wersję kodu (commit musi się zgadzać z CITATION.md):
git rev-parse HEAD
# 2. Zrekonstruuj środowisko:
conda env create -f appendix/F_environment/environment.yaml
# 3. Uruchom pełen sweep — Hydra multi-run z manifestem H_run_manifest.csv:
python main.py -m \
    optimizer=msffoa,ooa,ssa,nsga-3 \
    environment=forest,urban \
    seed=$(seq 1 30 | paste -sd, -) \
    simulation.gui=false
```

Każdy z 240 runów zajmuje ~3–15 min na maszynie referencyjnej
(AMD Ryzen 7 7700, 64 GB DDR5) — pełen sweep ~12–24 godz.